# SemMap — 4 независимых теста самоподобия nSBM (ConceptNet-10k)

Исследуем сохранённые уровни **22 → 7 → 3** из того же nSBM-прогона; не подменяем статистическую иерархию доказательством фрактальности.

1. **Структурный:** распределения направленных трёхвершинных мотивов и связь внутри/между блоками.
2. **Типизированный графон:** интенсивности отношений, JS распределений и диагностический held-out AUC.
3. **Хаккен:** ненулевые медленные моды линейной модели и углы между поднятыми спектральными подпространствами.
4. **Box-covering / мультифрактальность:** независимые радиусы исходного графа, степенной против экспоненциального закона, D(q).

**Нулевые модели:** перестановка вложенных групп в классах сходных входящих/исходящих степеней; разрыв пар концов рёбер отдельно по каждому типу, сохраняющий типизированные степени. Контрольные прогоны имеют отдельные checkpoints.

**Ограничения:** nSBM уже обучена на полном исходном графе, поэтому held-out оценка графона условная и не является независимым прогнозом; box-covering — эвристическая верхняя оценка числа коробок; Хаккен здесь линейный, не доказательство нелинейного подчинения. Для строгих выводов повторите несколько обучений nSBM и увеличьте число нулевых повторов.


In [ ]:
# 1. Конфигурация. При потере Colab-сессии сменить только RUN_MODE='RESUME'
from pathlib import Path
REPO_URL = "https://github.com/SemanticMap/semgraphex.git"
REPO_BRANCH = "feature/nsbm-multiscale-self-similarity"
SOURCE_REVISION = "875ef73bfce026f407489af0133cd81652fbda7b"  # fixed code SHA, not a moving branch
RUN_MODE = "NEW"  # NEW / RESUME, same source & scientific parameters
RUN_ID = "nsbm-selfsimilarity-cn10k-01"
SEED = 42
NULL_REPEATS = 19  # pilot; minimum attainable p=0.05. 99+ for confirmatory tests
CPU_BUDGET = 8
MOTIF_SAMPLES = 2500
PAIR_SAMPLES = 12000
BOX_REPEATS = 3
BOX_RADII = (0, 1, 2, 3, 4, 5)
SLOW_MODES = 4
SYNC_EVERY_SECONDS = 90
DRIVE_PROJECT = Path("/content/drive/MyDrive/SemanticMap")
PREVIOUS = DRIVE_PROJECT / "colab/hypothesis/runs/nsbm-huffman-10k-v3"
PERSISTENT = DRIVE_PROJECT / "colab/hypothesis/runs" / RUN_ID
assert RUN_MODE in ("NEW", "RESUME")


In [ ]:
# 2. Префлайт: CPU-бюджет, RAM, SSD, видеокарта (здесь GPU НЕ требуется)
import os,sys,json,shutil,subprocess,time,hashlib
def allowed_cpu():
    n=os.cpu_count() or 1
    if hasattr(os,"sched_getaffinity"):
        n=min(n,len(os.sched_getaffinity(0)))
    try:
        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if quota != "max":
            n=min(n,max(1,int((int(quota)+int(period)-1)//int(period))))
    except (OSError,ValueError): pass
    return max(1,n)
CPU_EFFECTIVE=min(CPU_BUDGET,allowed_cpu())
FREE_GB=shutil.disk_usage("/content").free/(1024**3)
print({"python":sys.version.split()[0],"cpu":allowed_cpu(),
       "workers":CPU_EFFECTIVE,"free_disk_gib":round(FREE_GB,2)})
if FREE_GB < 3: raise RuntimeError("Нужно минимум 3 GiB локального SSD")
# GPU/CuPy здесь сознательно не используются: CSR + 22/7/3 блоков на CPU быстрее.


In [ ]:
# 3. Google Drive → локальный SSD; сохраняем совпадение исходных IDs
from google.colab import drive
drive.mount("/content/drive")
SCRATCH=Path("/content/semmap_selfsim")
REPO=Path("/content/semmap_selfsim_repo")
GRAPH_DIR=SCRATCH/"fit_data"
PARTITIONS=SCRATCH/"nsbm_partitions.npz"
SOURCE_MANIFEST=SCRATCH/"source_nsbm_manifest.json"
RUN_LOCAL=SCRATCH/"runs"/RUN_ID
SCRATCH.mkdir(parents=True,exist_ok=True)
for name in ("ids.json","source.npy","target.npy","relation.npy","weight.npy"):
    inp=PREVIOUS/"fit_data"/name
    if not inp.is_file(): raise FileNotFoundError(f"Missing previous H1 artifact: {inp}")
    GRAPH_DIR.mkdir(parents=True,exist_ok=True)
    shutil.copy2(inp,GRAPH_DIR/name)
for src,dst in [(PREVIOUS/"nsbm_partitions.npz",PARTITIONS),
                (PREVIOUS/"manifest.json",SOURCE_MANIFEST)]:
    if not src.is_file(): raise FileNotFoundError(src)
    shutil.copy2(src,dst)
print("Staged:",GRAPH_DIR,PARTITIONS)
if RUN_MODE=="RESUME":
    if not PERSISTENT.is_dir(): raise FileNotFoundError(PERSISTENT)
    shutil.copytree(PERSISTENT,RUN_LOCAL,dirs_exist_ok=True)
else:
    if RUN_LOCAL.exists() and any(RUN_LOCAL.iterdir()):
        raise RuntimeError("NEW requires empty local run; select new RUN_ID")
    if PERSISTENT.exists() and any(PERSISTENT.iterdir()):
        raise RuntimeError("NEW refuses to overwrite a previous Drive run")


In [ ]:
# 4. Закреплённая ревизия и минимальные зависимости, без micromamba/graph-tool
if not (REPO/".git").is_dir():
    subprocess.run(["git","clone","--depth","80","--branch",REPO_BRANCH,
                    REPO_URL,str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","80",
                    "origin",REPO_BRANCH],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SOURCE_REVISION],check=True)
HEAD=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
assert HEAD==SOURCE_REVISION
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[dev,notebook]",
                "threadpoolctl>=3"],check=True)
print("Code:", HEAD)


In [ ]:
# 5. Проверка соответствия сохранённого графа разбиениям
from semmap_haken.hypothesis.runner import _load_graph,_sha_bytes
from semmap_haken.hypothesis.selfsim_experiment import load_levels
g=_load_graph(GRAPH_DIR)
levels=load_levels(PARTITIONS,g.vertex_count)
source=json.loads(SOURCE_MANIFEST.read_text())
assert source["graph_sha256"]==_sha_bytes(g), "Partition graph and original identifiers do NOT match!"
print({"vertices":g.vertex_count,"assertions":g.edge_count,
       "relations":g.relation_names,"block_counts":[len(set(p)) for p in levels]})
# The real graph contains original assertion multiplicities/directions/types.


In [ ]:
# 6. Тесты перед длительной работой (включая повреждение и RESUME)
check_env=dict(os.environ,OPENBLAS_NUM_THREADS="1",MKL_NUM_THREADS="1",
               OMP_NUM_THREADS="1",NUMEXPR_NUM_THREADS="1",PYTHONHASHSEED="0")
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_selfsim_four.py"],
               cwd=REPO,env=check_env,check=True)


In [ ]:
# 7. Длительная работа: локально считаем, готовые checkpoints копируем в Drive.
# Для RESUME запускаем CLI с теми же seed, parameters и закреплённым git SHA.
def sync_to_drive(final=False):
    PERSISTENT.mkdir(parents=True,exist_ok=True)
    for src in sorted(RUN_LOCAL.rglob("*")):
        if not src.is_file() or src.name=="COMPLETED" or src.name.endswith(".tmp"):
            continue
        dst=PERSISTENT/src.relative_to(RUN_LOCAL)
        dst.parent.mkdir(parents=True,exist_ok=True)
        if (not dst.exists() or dst.stat().st_size!=src.stat().st_size or
            dst.stat().st_mtime_ns!=src.stat().st_mtime_ns):
            shutil.copy2(src,dst)
    if final and (RUN_LOCAL/"COMPLETED").is_file():
        shutil.copy2(RUN_LOCAL/"COMPLETED",PERSISTENT/"COMPLETED")
cmd=[sys.executable,"-m","semmap_haken.hypothesis.selfsim_experiment",
     "--graph-dir",str(GRAPH_DIR),"--partitions",str(PARTITIONS),
     "--source-manifest",str(SOURCE_MANIFEST),"--output",str(RUN_LOCAL),
     "--mode",RUN_MODE,"--seed",str(SEED),"--null-repeats",str(NULL_REPEATS),
     "--jobs",str(CPU_EFFECTIVE),"--motif-samples",str(MOTIF_SAMPLES),
     "--pair-samples",str(PAIR_SAMPLES),"--box-repeats",str(BOX_REPEATS),
     "--slow-modes",str(SLOW_MODES),
     "--box-radii",",".join(map(str,BOX_RADII))]
LOG=SCRATCH/(RUN_ID+".log")
with LOG.open("w",encoding="utf-8") as log:
    proc=subprocess.Popen(cmd,cwd=REPO,env=check_env,
                          stdout=log,stderr=subprocess.STDOUT)
    try:
        while True:
            try:
                proc.wait(timeout=SYNC_EVERY_SECONDS)
                break
            except subprocess.TimeoutExpired:
                sync_to_drive()
                print("Partial checkpoints synced; completed null replicates:",
                      len(list(RUN_LOCAL.glob("null_*.json"))))
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        raise
    finally:
        sync_to_drive(final=proc.poll()==0)
if proc.returncode:
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-70:]))
    raise RuntimeError(f"Self-similarity CLI failed: {proc.returncode}")
assert (PERSISTENT/"COMPLETED").exists()
print("\n".join(LOG.read_text(errors="replace").splitlines()[-25:]))


In [ ]:
# 8. Сводка четырёх тестов и чувствительности к нулевым моделям
import pandas as pd
result=json.loads((PERSISTENT/"report.json").read_text(encoding="utf-8"))
print("Blocks:",result["block_counts"],"CPU workers:",result["cpu_workers"])
rows=[]
for metric,value in result["scores"].items():
    row={"metric":metric,"observed":value}
    for control,stats in result["controls"].items():
        x=stats.get(metric,{})
        row[control+"_null_mean"]=x.get("null_mean")
        row[control+"_p_upper"]=x.get("p_upper")
    rows.append(row)
display(pd.DataFrame(rows).round(5))
print("Independent predictive validation:",result["inference_warning"])
print("Fractal caveat:",result["scaling_warning"])
print("Saved report:",PERSISTENT/"report.json")


In [ ]:
# 9. Четыре отдельных графика (без ручной подгонки осей и цветов)
import matplotlib.pyplot as plt
observed=result["observed"]
s=observed["structural"]["adjacent_motif_js"]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot([f'{x["from"]}→{x["to"]}' for x in s],
        [x["js_divergence"] for x in s],marker="o")
ax.set(ylabel="JS divergence (bits)",title="1. Directed wedge motifs across scales")
ax.tick_params(axis="x",rotation=25)
plt.tight_layout();plt.show()
k=observed["graphon"]["adjacent_kernel"]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot([f'{x["fine_blocks"]}→{x["coarse_blocks"]}' for x in k],
        [x["log_intensity_js"] for x in k],marker="o")
ax.set(ylabel="JS divergence (bits)",title="2. Typed step-graphon intensity profiles")
plt.tight_layout();plt.show()
dy=observed["haken"]["levels"]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot([str(x["blocks"]) for x in dy],
        [x["slow_subspace_overlap"] for x in dy],marker="o")
ax.set(xlabel="nSBM blocks",ylabel="Mean squared principal cosine",
       ylim=(0,1.05),title="3. Slow-subspace preservation — Haken linear proxy")
plt.tight_layout();plt.show()
box=observed["boxes"]["rows"]
radii=sorted({x["radius"] for x in box})
xx=[2*r+1 for r in radii]
yy=[sum(x["box_count"] for x in box if x["radius"]==r)/BOX_REPEATS for r in radii]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.loglog(xx,yy,marker="o")
ax.set(xlabel="Greedy box scale (2r+1)",ylabel="N_boxes",
       title="4. Box-covering upper bound (not a fractal proof)")
plt.tight_layout();plt.show()


### Как интерпретировать результаты

- **Структурное подобие:** низкий JS мотивов полезен, только если существенно превосходит shuffled/nested controls. Для 3 блоков статистика трёхвершинных мотивов очень ограничена.
- **Графон:** сравнивайте нормированные по массам и типам отношений интенсивности и отложенные связи. AUC помечен как *условный*, потому что nSBM уже видела исходную сеть при обучении.
- **Хаккен:** оцениваем углы медленных подпространств, а не совпадение имён вершин. Модель здесь симметризована и линейна; динамика типов отношений и нелинейность не проверяются.
- **Box-covering:** показаны верхние оценки N_boxes(l), степенной vs экспоненциальный закон по AIC, D(q). При шести масштабах нельзя объявлять фрактальность; нужна устойчивость к радиусам и нулевым сетям.
- При **NULL_REPEATS=19** минимально возможное эмпирическое p равно 0,05. Для убедительной проверки выполните новый RUN_ID с NULL_REPEATS=99+ (RESUME не позволяет менять параметры).
- Данные и модель должны совпадать по SHA-256; все результаты сохраняются в Google Drive. Не переименовывайте старый RUN_ID с nSBM-Huffman.
